In [ ]:
import math
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.offsetbox import TextArea, VPacker, AnchoredOffsetbox

from pyproj import Geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LongitudeFormatter, LatitudeFormatter

from scipy.ndimage import gaussian_filter1d
from sklearn.cluster import DBSCAN
from statsmodels.nonparametric.smoothers_lowess import lowess

from pandas import Timestamp

In [ ]:
###################################-- initializations

targets = ["karachi-147b-pak-uhslc_rq",
           "maasvlakte_amaliahaven-maa-nld-rws",
           "booby_island-336-aus-uhslc_fd"]

jason_dirs  = [Path("jason1_hf"), Path("jason2_hf"), Path("jason3_f_hf")]

cutoff             = pd.Timestamp("2002-01-01")
window             = 24 * 7
std_threshold_abs  = 0.001
flatness_threshold = 0.5

geod = Geod(ellps="WGS84")

db_eps         = 0.04
db_minsamples  = 8
gaussian = 1.5
lowess_frac    = 0.2

#GIVE A SAMPLE GTSM FILE FOR SET UP#############################################
#example: _gtsm_setup_file = "GTSM_ERA5_2013_2019/reanalysis_surge_hourly_2013_10_v3.nc"
_gtsm_setup_file = "GTSM_ERA5_2013_2019/reanalysis_surge_hourly_2013_10_v3.nc"
_ds = xr.open_dataset(_gtsm_setup_file)
gtsm_station_lats = _ds["station_y_coordinate"].values
gtsm_station_lons = _ds["station_x_coordinate"].values
_ds.close()

#GIVE THE GTSM FOLDER NAME (IN THE PROJECT) #############################################
gtsmfolder = "GTSM_ERA5_2013_2019"

#GIVE THE LOCATION FOR THE TIDE GAUGES FOLDER#############################################
tgs = Path("/stormsurge/surge_comp")

In [ ]:
###################################-- tide gauge - altimetry matching, extreme events (storm surge) dictionary compilation

check_dist  = True

gauges = {}
for name in targets:
    fp = tgs / (name + "_surge.parquet")
    if not fp.exists():
        continue

    df = pd.read_parquet(fp, columns=["Latitude", "Longitude", "Time", "StormSurge"])
    df["Time"] = pd.to_datetime(df["Time"])

    if df["Time"].max() < cutoff:
        continue

    surge = df["StormSurge"].values
    times = df["Time"].values

    if len(df) < window * 2 or np.sum(~np.isnan(surge)) < window:
        continue

    rolling_std = pd.Series(surge).rolling(window, center=True, min_periods=window // 2).std()
    flat_mask   = (rolling_std < std_threshold_abs).fillna(False).values
    groups      = (flat_mask != np.roll(flat_mask, 1)).cumsum()
    seg_len     = pd.Series(flat_mask).groupby(groups).transform("size")
    long_flat   = flat_mask & (seg_len >= window)

    if np.mean(long_flat) > flatness_threshold:
        continue

    valid_mask  = ~long_flat & ~np.isnan(surge)
    valid_surge = surge[valid_mask]
    valid_times = times[valid_mask]
    if len(valid_surge) == 0:
        continue

    thr = valid_surge.mean() + 3 * valid_surge.std()
    ext_dates = pd.to_datetime(valid_times[valid_surge > thr]).normalize().unique()

    gauges[name] = {
        "lat": float(df["Latitude"].iloc[0]),
        "lon": float(df["Longitude"].iloc[0]),
        "adb_passes": [],
        "extreme_dates": [d.strftime("%Y-%m-%d") for d in sorted(ext_dates)],
        "three_sigma_threshold": float(thr),
    }

cycle_dirs = sorted(d for d in Path("jason2_hf").glob("cycle*") if d.is_dir())[:2]

for cdir in cycle_dirs:
    cycle_num = cdir.name.replace("cycle", "")
    for pass_file in sorted(cdir.glob("*_p[0-9][0-9][0-9][0-9].nc")):
        pass_num = pass_file.stem.split("_p")[-1]
        try:
            with xr.open_dataset(pass_file) as ds:
                p_lats = ds["lat_1hz"].values.ravel()
                p_lons = ds["lon_1hz"].values.ravel()
        except Exception:
            continue

        ok = np.isfinite(p_lats) & np.isfinite(p_lons)
        p_lats, p_lons = p_lats[ok], ((p_lons[ok] + 180) % 360) - 180
        if p_lats.size == 0:
            continue

        for name, g in gauges.items():
            g_lon = ((g["lon"] + 180) % 360) - 180
            _, _, dist = geod.inv(np.full_like(p_lons, g_lon),
                                  np.full_like(p_lats, g["lat"]),
                                  p_lons, p_lats)
            dmin = dist.min() / 1000.0
            if dmin <= 50:
                g["adb_passes"].append({
                    "cycle": cycle_num,
                    "pass": pass_num,
                    "path": str(pass_file),
                    "min_dist_km": float(dmin),
                })

gauges = {k: v for k, v in gauges.items() if v["adb_passes"]}

pass_cache = {}

for name, g in gauges.items():
    g_lon     = ((g["lon"] + 180) % 360) - 180
    pass_nums = sorted({p["pass"] for p in g["adb_passes"]})

    found     = {}

    for jdir in jason_dirs:
        for cdir in sorted(d for d in jdir.glob("cycle*") if d.is_dir()):
            cnum = cdir.name.replace("cycle", "")
            for pnum in pass_nums:
                fp = cdir / f"{jdir.name}_c{cnum}_p{pnum}.nc"
                if not fp.exists():
                    continue

                key = str(fp)
                if key not in pass_cache:
                    info = None
                    try:
                        with xr.open_dataset(fp, decode_times=False) as ds:
                            t   = ds["time_1hz"].values.ravel().astype("float64")
                            lat = ds["lat_1hz"].values.ravel().astype("float64")
                            lon = ds["lon_1hz"].values.ravel().astype("float64")
                        ok = np.isfinite(t) & (t > 0) & (t < 20000) & np.isfinite(lat) & np.isfinite(lon)
                        t, lat, lon = t[ok], lat[ok], ((lon[ok] + 180) % 360) - 180
                        if t.size:
                            info = {
                                "start": pd.Timestamp("2000-01-01 12:00:00") + pd.to_timedelta(float(t.min()), unit="D"),
                                "mid"  : pd.Timestamp("2000-01-01 12:00:00") + pd.to_timedelta(float(np.median(t)), unit="D"),
                                "end"  : pd.Timestamp("2000-01-01 12:00:00") + pd.to_timedelta(float(t.max()), unit="D"),
                                "lat"  : lat,
                                "lon"  : lon,
                            }
                    except Exception:
                        info = None
                    pass_cache[key] = info

                info = pass_cache[key]
                if info is None:
                    continue

                dmin = np.nan
                if check_dist:
                    _, _, d = geod.inv(np.full_like(info["lon"], g_lon),
                                       np.full_like(info["lat"], g["lat"]),
                                       info["lon"], info["lat"])
                    dmin = d.min() / 1000.0
                    if dmin > 50:
                        continue

                date = info["mid"].strftime("%Y-%m-%d")

                found[(date, pnum)] = {
                    "mission"    : jdir.name,
                    "cycle"      : cnum,
                    "pass"       : pnum,
                    "path"       : str(fp),
                    "time_start" : info["start"].strftime("%Y-%m-%d %H:%M:%S"),
                    "time_mid"   : info["mid"].strftime("%Y-%m-%d %H:%M:%S"),
                    "time_end"   : info["end"].strftime("%Y-%m-%d %H:%M:%S"),
                    "min_dist_km": float(dmin),
                }

    by_date = {}
    for (date, pnum), entry in sorted(found.items()):
        by_date.setdefault(date, []).append(entry)
    g["adb_pass_times"] = by_date

    g["adb_extreme_matches"] = {d: by_date[d] for d in g["extreme_dates"] if d in by_date}

    print(f"{name}")
    print(f"    extreme dates (with altimetry): {len(g['adb_extreme_matches'])} of {len(g['extreme_dates'])}")

print("matching and dictionary complete")

###################################

In [ ]:
###################################-- storm surge analysis
# set plotting option --> 0 = exclude plotting; 1 = include plotting

plotting = 1

results = {}

start_time = "2002-01-01"
end_time   = "2020-01-01"

for gauge_name, gauge_info in gauges.items():

    tidegauge_file  = gauge_name
    target_lon = gauge_info["lon"]
    target_lat = gauge_info["lat"]

    target_start = pd.Timestamp(start_time)
    target_end   = pd.Timestamp(end_time)

    df_gauge = pd.read_parquet(Path("/Users/jemmajohnson/Documents/MATLAB/surge_comp") / (gauge_name + "_surge.parquet"), columns=["Time", "StormSurge"])
    df_gauge["Time"] = pd.to_datetime(df_gauge["Time"])

    if df_gauge["Time"].max() < target_start or df_gauge["Time"].min() > target_end:
        continue

    three_sigma_threshold = gauge_info["three_sigma_threshold"]
    adb_extreme_matches   = gauge_info["adb_extreme_matches"]

    matching_days = sorted(
        pd.Timestamp(d) for d in adb_extreme_matches
        if target_start <= pd.Timestamp(d) <= target_end
    )

    if not matching_days:
        continue

    for ed in matching_days:
        date_str = ed.strftime("%Y-%m-%d")

        for entry in adb_extreme_matches[date_str]:

            adb_nc = Path(entry["path"])

            ds_jn          = xr.open_dataset(adb_nc)
            jn_lats_full   = ds_jn["lat_1hz"].values.ravel()
            jn_lons_full   = ds_jn["lon_1hz"].values.ravel()
            jn_sla_full    = ds_jn["sla_ALES_1hz"].values.ravel()
            jn_dac_full    = ds_jn["dac_1hz"].values.ravel()
            jn_time_full   = ds_jn["time_1hz"].values.ravel()
            ds_jn.close()

            _, _, jn_dist = geod.inv(
                np.full_like(jn_lats_full, target_lon),
                np.full_like(jn_lats_full, target_lat),
                jn_lons_full, jn_lats_full
            )
            jn_mask = (jn_dist <= 50 * 1000) & ~np.isnan(jn_sla_full)

            if jn_mask.sum() == 0:
                continue

            jn_lats_plot = jn_lats_full[jn_mask]
            jn_lons_plot = jn_lons_full[jn_mask]
            jn_sla_dac   = jn_sla_full[jn_mask] + jn_dac_full[jn_mask]
            jn_time_plot = jn_time_full[jn_mask]

            jn_sort      = np.argsort(jn_lats_plot)
            jn_lats_plot = jn_lats_plot[jn_sort]
            jn_lons_plot = jn_lons_plot[jn_sort]
            jn_sla_dac   = jn_sla_dac[jn_sort]
            jn_time_plot = jn_time_plot[jn_sort]

            _jn_mean = np.nanmean(jn_sla_dac)
            _jn_std  = np.nanstd(jn_sla_dac)
            _jn_3sig = np.abs(jn_sla_dac - _jn_mean) <= 3 * _jn_std
            jn_lats_plot = jn_lats_plot[_jn_3sig]
            jn_lons_plot = jn_lons_plot[_jn_3sig]
            jn_sla_dac   = jn_sla_dac[_jn_3sig]
            jn_time_plot = jn_time_plot[_jn_3sig]

            if len(jn_time_plot) == 0:
                continue

            jn_mid       = jn_time_plot[len(jn_time_plot) // 2]
            jn_mid_time  = pd.Timestamp("2000-01-01 12:00:00") + pd.to_timedelta(float(jn_mid), unit="D")

            day_start  = pd.Timestamp(ed.date())
            day_end    = day_start + pd.Timedelta(days=1)
            mask_surge = (df_gauge['Time'] >= day_start) & (df_gauge['Time'] < day_end)

            surge_times = df_gauge['Time'][mask_surge]
            surge_vals  = df_gauge['StormSurge'][mask_surge]

            if len(surge_times) == 0:
                continue

            sla_time_on_event_day = pd.Timestamp(ed.date()) + pd.Timedelta(
                hours=jn_mid_time.hour,
                minutes=jn_mid_time.minute,
                seconds=jn_mid_time.second
            )
            surge_at_sla_time = surge_vals.iloc[
                np.argmin(np.abs(surge_times - sla_time_on_event_day))
            ]

            tg_value        = float(surge_at_sla_time)
            tg_above_3sigma = int(tg_value > three_sigma_threshold)

            if not tg_above_3sigma:
                continue

            jn_valid = ~np.isnan(jn_sla_dac)
            jn_filt  = jn_sla_dac.copy()
            if jn_valid.sum() > db_minsamples:
                jn_db = DBSCAN(eps=db_eps, min_samples=db_minsamples).fit(
                    np.vstack([jn_lats_plot[jn_valid], jn_sla_dac[jn_valid]]).T)
                valid_positions   = np.where(jn_valid)[0]
                outlier_positions  = valid_positions[jn_db.labels_ == -1]
                jn_filt[outlier_positions] = np.nan

            jn_mask2 = ~np.isnan(jn_filt)
            if jn_mask2.sum() > 0:
                jn_filt[jn_mask2] = gaussian_filter1d(
                    jn_filt[jn_mask2], sigma= gaussian)

            jn_valid2 = ~np.isnan(jn_filt)
            if jn_valid2.sum() > 2:
                jn_bg    = lowess(jn_filt[jn_valid2], jn_lats_plot[jn_valid2],
                                frac=lowess_frac, return_sorted=False)
                jn_bg_full = np.interp(jn_lats_plot, jn_lats_plot[jn_valid2], jn_bg)
            else:
                jn_bg_full = None

            jn_raw_notnan  = ~np.isnan(jn_sla_dac)
            jn_filt_notnan = (~np.isnan(jn_bg_full)
                            if jn_bg_full is not None
                            else np.zeros(len(jn_lats_plot), dtype=bool))

            nlat_raw_lat  = np.nan
            nlat_raw_lon  = np.nan
            nlat_raw_val  = np.nan
            nlat_filt_lat = np.nan
            nlat_filt_lon = np.nan
            nlat_filt_val = np.nan

            if jn_raw_notnan.any():
                dl = np.abs(jn_lats_plot - target_lat)
                dl[~jn_raw_notnan] = np.inf
                nlat_raw_idx = np.argmin(dl)
                nlat_raw_lat = float(jn_lats_plot[nlat_raw_idx])
                nlat_raw_lon = float(jn_lons_plot[nlat_raw_idx])
                nlat_raw_val = float(jn_sla_dac[nlat_raw_idx])

            if jn_filt_notnan.any():
                dl = np.abs(jn_lats_plot - target_lat)
                dl[~jn_filt_notnan] = np.inf
                nlat_filt_idx = np.argmin(dl)
                nlat_filt_lat = float(jn_lats_plot[nlat_filt_idx])
                nlat_filt_lon = float(jn_lons_plot[nlat_filt_idx])
                nlat_filt_val = float(jn_bg_full[nlat_filt_idx])

            ntg_lat = nlat_filt_lat if not np.isnan(nlat_filt_lat) else nlat_raw_lat
            ntg_lon = nlat_filt_lon if not np.isnan(nlat_filt_lon) else nlat_raw_lon

            ntg_dist_km = np.nan
            if not np.isnan(ntg_lat):
                _, _, ntg_dist_m = geod.inv(target_lon, target_lat, ntg_lon, ntg_lat)
                ntg_dist_km = float(ntg_dist_m) / 1000

            results.setdefault(tidegauge_file, {})
            results[tidegauge_file][date_str] = {
                "surge_time"            : jn_mid_time.strftime("%Y:%m:%d:%H:%M"),
                "surge_at_sat_time"     : tg_value,
                "three_sigma"           : three_sigma_threshold,

                "adb_raw"               : nlat_raw_val,
                "adb_filt"              : nlat_filt_val,
                "adb_dist_to_gauge"     : ntg_dist_km,
                "closest_gauge_adb_lat" : ntg_lat,
                "closest_gauge_adb_lon" : ntg_lon,
            }

            if plotting == 1: 
                fig     = plt.figure(figsize=(13, 11))
                ax_sla  = plt.subplot(2, 2, 1)

                lab_raw  = (f"ADB (Unfiltered)"
                            if not np.isnan(nlat_raw_val) else "ADB (unfiltered)")
                lab_filt = (f"ADB (Filtered) | NTG = {nlat_filt_val:.3f}"
                            if not np.isnan(nlat_filt_val) else "ADB (filtered)")

                ax_sla.plot(jn_lats_plot, jn_sla_dac, label=lab_raw, linewidth=2, color="grey")
                if jn_bg_full is not None:
                    ax_sla.plot(jn_lats_plot, jn_bg_full, label=lab_filt, linewidth=2, color="black")

                if not np.isnan(nlat_raw_val):
                    ax_sla.plot(nlat_raw_lat, nlat_raw_val, 'o', color="grey", markeredgecolor="black",
                                markersize=9, zorder=10)
                if not np.isnan(nlat_filt_val):
                    ax_sla.plot(nlat_filt_lat, nlat_filt_val, 'o', color="black", markeredgecolor="white",
                                markersize=9, zorder=11)

                ax_sla.set_xlabel("Latitude (°)", fontsize=15, labelpad=10)
                ax_sla.set_ylabel("SLA+DAC (m)", fontsize=15)
                ax_sla.set_ylim(-0.5, 2.5)
                ax_sla.grid(True)
    
                ax_sla.set_title(f"a) SLA+DAC vs Latitude on {ed.date()}", fontsize=15)

                lines_sla,  labels_sla  = ax_sla.get_legend_handles_labels()
                ax_sla.legend(lines_sla, labels_sla, loc='lower right', fontsize=13)

                ax_sla.tick_params(axis='x', labelsize=15)
                ax_sla.tick_params(axis='y', labelsize=15)

                proj   = ccrs.PlateCarree()
                ax_map = fig.add_subplot(2, 2, 2, projection=proj)

                lon_min, lon_max = np.floor(target_lon - 1), np.ceil(target_lon + 1)
                lat_min, lat_max = np.floor(target_lat - 1), np.ceil(target_lat + 1)
                ax_map.set_extent([lon_min, lon_max, lat_min, lat_max], crs=proj)

                ax_map.add_feature(cfeature.LAND, facecolor="#e9eaf0",
                                    edgecolor="#868d96", linewidth=1.5, zorder=3)
                ax_map.add_feature(cfeature.COASTLINE)

                gl = ax_map.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
                gl.top_labels   = False
                gl.right_labels = False
                gl.xlabel_style = {'size': 15}
                gl.ylabel_style = {'size': 15}
                gl.xformatter   = LongitudeFormatter()
                gl.yformatter   = LatitudeFormatter()

                lon_ticks = np.arange(np.floor(lon_min), np.ceil(lon_max) + 1.5, 2)
                lat_ticks = np.arange(np.floor(lat_min), np.ceil(lat_max) + 1.5, 2)
                gl.xlocator = plt.FixedLocator(lon_ticks)
                gl.ylocator = plt.FixedLocator(lat_ticks)

                ax_map.plot(target_lon, target_lat, 'ro', markersize=7,
                            transform=proj, label='Tide Gauge', zorder=6)

                n_points = 100
                az = np.linspace(0, 360, n_points)
                lons_circle, lats_circle, _ = geod.fwd(
                    np.full(n_points, target_lon), np.full(n_points, target_lat),
                    az, np.full(n_points, 50 * 1000)
                )
                ax_map.plot(lons_circle, lats_circle, 'k--', transform=proj,
                            linewidth=1.5, label=f'{50} km radius', zorder=12)

                inside_mask = jn_dist <= 50 * 1000
                box_mask = ((jn_lons_full >= lon_min) & (jn_lons_full <= lon_max) &
                            (jn_lats_full >= lat_min) & (jn_lats_full <= lat_max))
                map_points_mask = inside_mask & box_mask
                if np.any(map_points_mask):
                    ax_map.scatter(jn_lons_full[map_points_mask], jn_lats_full[map_points_mask],
                                color="#2A63B7", s=15, transform=proj, zorder=2,
                                label="Altimetry Track")

                ax_map.tick_params(axis='x', labelsize=15)
                ax_map.tick_params(axis='y', labelsize=15)
                ax_map.set_title(f"b) {tidegauge_file}", fontsize=15)

                ax_map.legend(loc="lower right", fontsize=15)

                ax_surge = plt.subplot(2, 1, 2)

                if len(surge_times) > 0:
                    ax_surge.plot(surge_times, surge_vals, linewidth=2, color="#2A63B7", label='Surge')

                    ax_surge.axvline(sla_time_on_event_day, color='red', linestyle='--',
                                    linewidth=2, label='Satellite Pass Time')
                    ax_surge.annotate(
                        f"{surge_at_sla_time:.3f} m",
                        xy=(sla_time_on_event_day, surge_at_sla_time),
                        xytext=(12, 12), textcoords="offset points",
                        fontsize=15, color="red",
                        arrowprops=dict(arrowstyle="->", color="red", lw=1),
                        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7)
                    )

                    ax_surge.axhline(three_sigma_threshold, color='gray', linestyle='--',
                                    linewidth=2, label=f'3σ Threshold: {three_sigma_threshold:.3f} m')

                ax_surge.set_xlabel("Time", fontsize=15)
                ax_surge.set_title(f"c) Tide Gauge Surge on {ed.date()}", fontsize=15)
                ax_surge.set_ylabel("Surge (m)", fontsize=15)
                ax_surge.tick_params(axis='x', labelsize=15)
                ax_surge.tick_params(axis='y', labelsize=15)
                ax_surge.grid(True)
                ax_surge.legend(loc="lower left", fontsize=15)
                ax_surge.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
                plt.setp(ax_surge.xaxis.get_majorticklabels(), rotation=45, ha='right')
                #plt.savefig(f"{tidegauge_file}_{ed.date()}_p{entry['pass']}.png", dpi=300)
                plt.subplots_adjust(top=0.95, bottom=0.08, left=0.08, right=0.95, hspace=0.25, wspace=0.3)
                plt.show()
                plt.close(fig)

print("storm surge analysis complete")

###################################

In [ ]:
###################################-- storm surge results validation with GTSM
# compares the altimetry NTG points with the tide gauge measurement with the GTSM estimate

gtsm_points = {}

all_dates = []
all_tg = []
all_adb_raw = []
all_adb_filt = []
all_gtsm = []
all_gauge_ids = []

gauge_list = list(results.keys())
gauge_id_map = {g: i + 1 for i, g in enumerate(gauge_list)}
id_to_name = {v: k for k, v in gauge_id_map.items()}

for gauge_name, events in results.items():

    gauge_id = gauge_id_map[gauge_name]

    for date, vals in events.items():

        tg = vals.get("surge_at_sat_time", np.nan)

        if np.isnan(tg):
            continue

        ntg_lat = vals.get("closest_gauge_adb_lat", np.nan)
        ntg_lon = vals.get("closest_gauge_adb_lon", np.nan)
        gtsm_val = np.nan

        if not (np.isnan(ntg_lat) or np.isnan(ntg_lon)):
            ts    = pd.to_datetime(vals["surge_time"], format="%Y:%m:%d:%H:%M")
            year  = int(ts.year)
            month = int(ts.month)

            dist        = np.sqrt((gtsm_station_lons - ntg_lon)**2 + (gtsm_station_lats - ntg_lat)**2)
            nearest_idx = int(dist.argmin())

            fp = Path(gtsmfolder) / f"reanalysis_surge_hourly_{year}_{month:02d}_v3.nc"

            if fp.exists():
                key = (str(fp), nearest_idx)
                if key not in gtsm_points:
                    ds = xr.open_dataset(fp)
                    gtsm_points[key] = (pd.to_datetime(ds["time"].values),
                                       ds["surge"][:, nearest_idx].values)
                    ds.close()
                times_gtsm, surge_gtsm = gtsm_points[key]

                time_diffs    = np.abs(times_gtsm - ts)
                closest_t_idx = int(time_diffs.argmin())
                if time_diffs[closest_t_idx] <= pd.Timedelta(hours=3) and not np.isnan(surge_gtsm[closest_t_idx]):
                    gtsm_val = float(surge_gtsm[closest_t_idx])

        all_dates.append(date)
        all_tg.append(tg)
        all_gtsm.append(gtsm_val)

        all_adb_raw.append(vals.get("adb_raw", np.nan))
        all_adb_filt.append(vals.get("adb_filt", np.nan))

        all_gauge_ids.append(gauge_id)

n_chunks = math.ceil(len(all_dates) / 20)

for chunk_idx in range(n_chunks):

    start = chunk_idx * 20
    end = min((chunk_idx + 1) * 20, len(all_dates))

    dates_chunk = all_dates[start:end]

    tg_chunk = all_tg[start:end]

    gtsm_chunk = all_gtsm[start:end]

    adb_raw_chunk = all_adb_raw[start:end]
    adb_filt_chunk = all_adb_filt[start:end]

    gauge_ids_chunk = all_gauge_ids[start:end]

    x = np.arange(len(dates_chunk))
    width = 0.2

    fig, ax = plt.subplots(figsize=(16, 7))

    ax.bar(x - 1.5 * width, tg_chunk, width, color='red', label='Tide Gauge')
    ax.bar(x - 0.5 * width, gtsm_chunk, width, color='#3d78d1', label='GTSM')
    ax.bar(x + 0.5 * width, adb_filt_chunk, width, color='black', label='ADB Filtered')
    ax.bar(x + 1.5 * width, adb_raw_chunk, width, color='grey', label='ADB Unfiltered')

    ax.set_xticks(x)
    tick_labels = [f"[{gid}] {d}" for gid, d in zip(gauge_ids_chunk, dates_chunk)]
    ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=11)

    ax.set_ylabel("Sea Level (m)", fontsize=11)
    ax.set_title("Tide Gauge vs. Altimetry vs. GTSM", fontsize=11)

    ax.grid(True, axis='y', linestyle='--', alpha=0.5)
    ax.legend(fontsize=11)
    ax.tick_params(axis='both', labelsize=11)

    ids_sorted = sorted(set(gauge_ids_chunk))
    entries = [f"[{gid}] {id_to_name[gid]}" for gid in ids_sorted]
    lines = ["   ".join(entries[i:i + 3]) for i in range(0, len(entries), 3)]
    legend_text = "\n".join(lines)

    fig.subplots_adjust(bottom=0.22 + 0.03 * (len(lines) - 1))
    fig.text(0.5, 0.02, legend_text, ha='center', va='bottom', fontsize=11)

    plt.show()


gauge_adb_diffs = defaultdict(list)
gauge_n_events = defaultdict(int)

for gauge_name, events in results.items():
    for date, vals in events.items():
        tg = vals.get("surge_at_sat_time", np.nan)
        jason = vals.get("adb_filt", np.nan)
        if np.isnan(tg):
            continue
        gauge_n_events[gauge_name] += 1
        if not np.isnan(jason):
            gauge_adb_diffs[gauge_name].append(tg - jason)

rows = []
for gauge_name in results.keys():
    n = gauge_n_events[gauge_name]
    if n == 0:
        continue
    avg_adb = float(np.mean(gauge_adb_diffs[gauge_name])) if gauge_adb_diffs[gauge_name] else np.nan

    sort_key = abs(avg_adb) if not np.isnan(avg_adb) else np.inf
    rows.append((gauge_name, n, avg_adb, sort_key))

rows.sort(key=lambda r: r[-1])

plot_adb_diffs = [r[2] for r in rows]
plot_gauge_names = [r[0] for r in rows]

print("gauge-altimetry validation complete")

###################################